# Get 2D Fingerprints of All Molecules in COCONUT

## 1. Setup, imports & data loading

In [ ]:
from joblib import delayed, Parallel
import pandas as pd
from contextlib import contextmanager
import joblib
from tqdm.notebook import tqdm

from fingerprint_utils.twod_fingerprint import ecfp_from_smiles, pharmacophore_fp_from_smiles, maccs_fp_from_smiles
from similarity_utils.fingerprint_similarities import tanimoto_similarity, fp_from_bitstring

In [ ]:
df = pd.read_csv('/path/to.csv')
df.head()

# smaller subset of df for testing
# working_df = df.sample(n=50, random_state=42).reset_index(drop=True)

## 2. Generate fingerprints for molecules

Start by taking reference molecule fingerprints from acarbose:

In [ ]:
ACARBOSE_SMILES = "C[C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)O[C@@H]2[C@H](O[C@@H]([C@@H]([C@H]2O)O)O[C@H]([C@@H](CO)O)[C@@H]([C@H](C=O)O)O)CO)O)O)N[C@H]3C=C([C@H]([C@@H]([C@H]3O)O)O)CO"

acarbose_ecfp = ecfp_from_smiles(ACARBOSE_SMILES)
acarbose_pharmacophore_fp = pharmacophore_fp_from_smiles(ACARBOSE_SMILES)
acarbose_maccs_fp = maccs_fp_from_smiles(ACARBOSE_SMILES)

In [ ]:
@contextmanager
def tqdm_joblib(tqdm_object):
    """
    Context manager to patch joblib's BatchCompletionCallBack to report into tqdm.
    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=1)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

# ECFP
with tqdm_joblib(tqdm(total=len(df), desc="ECFP")):
    df['ecfp'] = Parallel(n_jobs=-1)(delayed(ecfp_from_smiles)(smiles) for smiles in df['normalized_smiles'])
print("ECFP generation complete.")

# Pharmacophore
with tqdm_joblib(tqdm(total=len(df), desc="Pharmacophore FP")):
    df['pharmacophore_fp'] = Parallel(n_jobs=-1)(delayed(pharmacophore_fp_from_smiles)(smiles) for smiles in df['normalized_smiles'])
print("Pharmacophore fingerprint generation complete.")

# MACCS
with tqdm_joblib(tqdm(total=len(df), desc="MACCS FP")):
    df['maccs_fp'] = Parallel(n_jobs=-1)(delayed(maccs_fp_from_smiles)(smiles) for smiles in df['normalized_smiles'])
print("MACCS fingerprint generation complete.")

ECFP:   0%|          | 0/1000 [00:00<?, ?it/s]

ECFP generation complete.


Pharmacophore FP:   0%|          | 0/1000 [00:00<?, ?it/s]

Pharmacophore fingerprint generation complete.


MACCS FP:   0%|          | 0/1000 [00:00<?, ?it/s]

MACCS fingerprint generation complete.


## 3. Calculate similarities between molecules and acarbose

In [ ]:
# ECFP similarity
with tqdm_joblib(tqdm(total=len(df), desc="ECFP Similarity")):
    df['ecfp_similarity'] = Parallel(n_jobs=-1)(delayed(lambda x: tanimoto_similarity(fp_from_bitstring(x), fp_from_bitstring(acarbose_ecfp)))(fp) for fp in df['ecfp'])
print("ECFP similarity calculation complete.")

with tqdm_joblib(tqdm(total=len(df), desc="Pharmacophore Similarity")):
    df['pharmacophore_similarity'] = Parallel(n_jobs=-1)(delayed(lambda x: tanimoto_similarity(fp_from_bitstring(x), fp_from_bitstring(acarbose_pharmacophore_fp)))(fp) for fp in df['pharmacophore_fp'])
print("Pharmacophore similarity calculation complete.")

# MACCS similarity
with tqdm_joblib(tqdm(total=len(df), desc="MACCS Similarity")):
    df['maccs_similarity'] = Parallel(n_jobs=-1)(delayed(lambda x: tanimoto_similarity(fp_from_bitstring(x), fp_from_bitstring(acarbose_maccs_fp)))(fp) for fp in df['maccs_fp'])
print("MACCS similarity calculation complete.")

0


In [ ]:
df.to_csv('coconut_with_2d_fps.csv')